In [1]:
from google.colab import drive
drive.mount('/content/drive',  force_remount=True)
%cd '/content/drive/Othercomputers/NDL Desktop/Desktop/Research_crimeWarnerRobins'

Mounted at /content/drive
/content/drive/Othercomputers/NDL Desktop/Desktop/Research_crimeWarnerRobins


In [2]:
import google.generativeai as genai
from google.colab import userdata

# Get the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini Generative Model
model = genai.GenerativeModel('gemini-2.5-flash')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
import os
import pandas as pd
import json
import google.generativeai as genai

def print_file_names(folder_name):
    try:
        files = os.listdir(folder_name)
        for file in files:
            print(file)
    except FileNotFoundError:
        print(f"Folder '{folder_name}' not found.")
    except PermissionError:
        print(f"Permission denied for folder '{folder_name}'.")


def read_folder_to_df_xlsx(folder_path, iter_limit=None):
    """
    Reads all .xlsx files from a specified folder into pandas DataFrames,
    combines them, filters columns based on Gemini's identification of
    'occurrence time' and 'coordinates', renames them, ensures consistent order,
    and removes duplicated rows.

    Args:
        folder_path (str): The path to the folder containing the .xlsx files.
        iter_limit (int, optional): The maximum number of .xlsx files to process.
                                    If None, all files will be processed. Defaults to None.

    Returns:
        pandas.DataFrame: A single pandas DataFrame containing combined and cleaned data.
                          Returns an empty DataFrame if no files are processed or an error occurs.
    """
    dataframes = []
    files_processed_count = 0
    try:
        if 'model' not in globals():
            print("Gemini model not initialized. Please ensure the API key is configured and the model is loaded.")
            return pd.DataFrame() # Return empty DataFrame if model not initialized

        for filename in os.listdir(folder_path):
            if iter_limit is not None and files_processed_count >= iter_limit:
                print(f"Iteration limit of {iter_limit} reached. Stopping file processing.")
                break

            if filename.endswith('.xlsx'):
                file_path = os.path.join(folder_path, filename)
                print(f"\nReading file: {filename}")
                df = pd.read_excel(file_path)
                print(f"  Original number of rows: {df.shape[0]}")
                original_columns = df.columns.tolist()
                print(f"  Original column names: {original_columns}")

                # Use Gemini to identify relevant columns
                prompt = f"I have a list of column names from a DataFrame: {original_columns}. Please identify the column names that represent 'occurrence time of row data' and 'coordinates of row data'. If there are multiple datetime col, select the \"time of occurence\". Column names may have abbreviations. Return the identified column names as a JSON list. The first element should be the datetime column, followed by the two coordinate columns (e.g., latitude, longitude or x, y). Do not include any other text, only the JSON list. For example, if 'date_occu' is occurrence time and 'geox', 'geoy' are coordinates, return: [\"date_occu\", \"geox\", \"geoy\"]"

                print(f"  Calling Gemini for column identification for {filename}...")
                identified_cols_raw = []
                try:
                    response = model.generate_content(prompt)
                    response_text = response.text.strip().replace('```json', '').replace('```', '')
                    identified_cols_raw = json.loads(response_text)
                    print(f"  Columns identified by Gemini: {identified_cols_raw}")
                except Exception as e:
                    print(f"  Error calling Gemini or parsing response for {filename}: {e}")
                    print(f"  Gemini's raw response (if available): {response.text if 'response' in locals() else 'No response'}")
                    print("  Proceeding without filtering and renaming columns for this file.")
                    # If identification fails, do not append this df, as we can't ensure consistency for combined DF
                    print(f"  Finished processing {filename}.")
                    files_processed_count += 1
                    continue # Skip to next file

                datetime_col = None
                coord_cols = []

                # Try to map identified_cols_raw to datetime and coordinates
                if identified_cols_raw:
                    # Assuming the first is datetime, and the next two are coordinates based on prompt example
                    if len(identified_cols_raw) > 0:
                        datetime_col = identified_cols_raw[0]
                    if len(identified_cols_raw) > 1:
                        coord_cols = identified_cols_raw[1:3] # Take up to two coordinate columns

                # Assertions for required columns
                try:
                    assert datetime_col is not None and datetime_col in original_columns, f"Could not identify a valid 'occurrence time' column in {filename}."
                    assert len(coord_cols) == 2 and all(c in original_columns for c in coord_cols), f"Could not identify exactly two valid coordinate columns in {filename}."
                except AssertionError as e:
                    print(f"  Assertion failed for {filename}: {e}")
                    print("  Proceeding without filtering and renaming, keeping original columns for this file.")
                    # If assertion fails, do not append this df, as we can't ensure consistency for combined DF
                    print(f"  Finished processing {filename}.")
                    files_processed_count += 1
                    continue # Skip to next file

                # Filter and rename columns
                cols_to_keep_and_rename = {
                    datetime_col: 'datetime_occu',
                    coord_cols[0]: 'x',
                    coord_cols[1]: 'y'
                }

                # Create new df with only the required columns, in the desired order
                df_filtered = df[[datetime_col, coord_cols[0], coord_cols[1]]].copy()
                df_filtered.rename(columns=cols_to_keep_and_rename, inplace=True)
                print(f"  Columns filtered and renamed: {list(df_filtered.columns)}")

                # Convert 'datetime_occu' column to datetime format
                try:
                    df_filtered['datetime_occu'] = pd.to_datetime(df_filtered['datetime_occu'], errors='coerce')
                    print(f"  Converted 'datetime_occu' to datetime format.")
                except Exception as e:
                    print(f"  Error converting 'datetime_occu' to datetime in {filename}: {e}")

                # Sort by datetime_occu (this will be done on the combined DF later)
                # df_filtered.sort_values(by='datetime_occu', inplace=True)
                # print(f"  DataFrame sorted by 'datetime_occu'.")

                # Treat 0 coordinates as missing data
                for col in ['x', 'y']:
                    if col in df_filtered.columns:
                        # Replace 0 values with pandas' native missing value NA
                        df_filtered[col] = df_filtered[col].replace(0, pd.NA)
                print("  Replaced 0 in 'x' and 'y' coordinates with NA for missing data handling.")

                # Remove rows with missing data (NaNs) and print them before deletion
                initial_rows = df_filtered.shape[0]
                rows_with_nan = df_filtered[df_filtered.isnull().any(axis=1)]
                if not rows_with_nan.empty:
                    print(f"  Found {rows_with_nan.shape[0]} rows with missing data. These rows will be dropped:")
                    # print(rows_with_nan)
                    df_filtered.dropna(inplace=True)
                    print(f"  Dropped {initial_rows - df_filtered.shape[0]} rows with missing data.")
                else:
                    print("  No missing data found in this DataFrame.")


                # print(f"  Head of processed {filename}:")
                # print(df_filtered.head())

                dataframes.append(df_filtered)
                print(f"  Finished processing {filename}.")
                files_processed_count += 1
    except FileNotFoundError:
        print(f"Error: Folder '{folder_path}' not found.")
        return pd.DataFrame() # Return empty DataFrame on error
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return pd.DataFrame() # Return empty DataFrame on error

    # Combine all dataframes into a single DataFrame
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        print(f"\nCombined all {len(dataframes)} DataFrames. Total rows: {combined_df.shape[0]}")

        # Remove duplicate rows
        initial_combined_rows = combined_df.shape[0]
        combined_df.drop_duplicates(inplace=True)
        if combined_df.shape[0] < initial_combined_rows:
            print(f"Removed {initial_combined_rows - combined_df.shape[0]} duplicate rows.")
        else:
            print("No duplicate rows found in the combined DataFrame.")

        # Sort the final combined DataFrame by datetime_occu
        combined_df.sort_values(by='datetime_occu', inplace=True)
        print("Combined DataFrame sorted by 'datetime_occu'.")

        return combined_df
    else:
        print("No valid DataFrames were processed and combined.")
        return pd.DataFrame() # Return empty DataFrame if no dataframes were appended


In [6]:
# print_file_names('./data/data_after_move_stops')
combined_df = read_folder_to_df_xlsx('./data/data_after_move_stops')
if not combined_df.empty:
    print("\nHead of the combined, cleaned, and deduplicated DataFrame:")
    display(combined_df.head())
    print(f"Total rows in combined DataFrame: {combined_df.shape[0]}")
else:
    print("No DataFrame was generated.")

output_path = './data/data_after_move_stops.csv'
combined_df.to_csv(output_path, index=False)
print(f"DataFrame saved to {output_path}")


Reading file: GT 102323.xlsx
  Original number of rows: 117
  Original column names: ['date_fnd', 'date_occu', 'date_rept', 'geox', 'geoy', 'inci_id', 'street', 'streetnbr', 'groupa']
  Calling Gemini for column identification for GT 102323.xlsx...
  Columns identified by Gemini: ['date_occu', 'geox', 'geoy']
  Columns filtered and renamed: ['datetime_occu', 'x', 'y']
  Converted 'datetime_occu' to datetime format.
  Replaced 0 in 'x' and 'y' coordinates with NA for missing data handling.
  Found 6 rows with missing data. These rows will be dropped:
  Dropped 6 rows with missing data.
  Finished processing GT 102323.xlsx.

Reading file: GT county wide 102323.xlsx
  Original number of rows: 181
  Original column names: ['date_fnd', 'date_occu', 'date_rept', 'geox', 'geoy', 'inci_id', 'street', 'streetnbr', 'groupa']
  Calling Gemini for column identification for GT county wide 102323.xlsx...
  Columns identified by Gemini: ['date_occu', 'geox', 'geoy']
  Columns filtered and renamed: [

,datetime_occu,x,y
877,2023-10-01 00:00:00,2463458.25,952822.88
876,2023-10-01 01:00:00,2459639.75,951314.87
827,2023-10-01 01:25:00,2463439.25,948953.63
829,2023-10-01 01:30:00,2460463.75,949959.5
853,2023-10-01 02:00:59,2455795.5,953056.31


Total rows in combined DataFrame: 3517
DataFrame saved to ./data/data_after_move.csv


In [7]:
combined_df

,datetime_occu,x,y
877,2023-10-01 00:00:00,2463458.25,952822.88
876,2023-10-01 01:00:00,2459639.75,951314.87
827,2023-10-01 01:25:00,2463439.25,948953.63
829,2023-10-01 01:30:00,2460463.75,949959.5
853,2023-10-01 02:00:59,2455795.5,953056.31
...,...,...,...
4617,2024-09-08 13:01:00,2467318.0,942475.44
4616,2024-09-08 14:50:21,2459356.75,940627.5
4619,2024-09-08 16:00:59,2469077.75,943256.31
4618,2024-09-08 16:34:59,2459634.25,963149.56
